In [1]:
import pandas as pd
import time
from pathlib import Path

DATA_DIR = Path.home() / "projects" / "recsys" / "data" / "ml-25m"
OUT_DIR = Path.home() / "projects" / "recsys" / "data" / "parquet"
OUT_DIR.mkdir(exist_ok=True)

print("pandas:", pd.__version__)
print("data dir:", DATA_DIR)
print("files:", [f.name for f in DATA_DIR.glob("*.csv")])

pandas: 3.0.2
data dir: /Users/nitishpatil/projects/recsys/data/ml-25m
files: ['links.csv', 'tags.csv', 'genome-tags.csv', 'ratings.csv', 'genome-scores.csv', 'movies.csv']


In [2]:
files = ["movies.csv", "links.csv", "tags.csv", "genome-tags.csv", "ratings.csv", "genome-scores.csv"]

for fname in files:
    csv_path = DATA_DIR / fname
    parquet_path = OUT_DIR / fname.replace(".csv", ".parquet")
    
    t0 = time.time()
    df = pd.read_csv(csv_path)
    t_read = time.time() - t0
    
    t0 = time.time()
    df.to_parquet(parquet_path, compression="snappy")
    t_write = time.time() - t0
    
    csv_size = csv_path.stat().st_size / 1e6
    parquet_size = parquet_path.stat().st_size / 1e6
    
    print(f"{fname:25s} | rows: {len(df):>10,} | csv: {csv_size:>7.1f} mb | parquet: {parquet_size:>7.1f} mb | read: {t_read:>5.1f}s | write: {t_write:>5.1f}s")

print("\ndone")

movies.csv                | rows:     62,423 | csv:     3.0 mb | parquet:     1.8 mb | read:   0.0s | write:   1.7s
links.csv                 | rows:     62,423 | csv:     1.4 mb | parquet:     1.2 mb | read:   0.0s | write:   0.0s
tags.csv                  | rows:  1,093,360 | csv:    38.8 mb | parquet:    10.9 mb | read:   0.2s | write:   0.1s
genome-tags.csv           | rows:      1,128 | csv:     0.0 mb | parquet:     0.0 mb | read:   0.0s | write:   0.0s
ratings.csv               | rows: 25,000,095 | csv:   678.3 mb | parquet:   166.6 mb | read:   2.5s | write:   0.9s
genome-scores.csv         | rows: 15,584,448 | csv:   435.2 mb | parquet:    26.8 mb | read:   1.4s | write:   0.3s

done


In [3]:
print("comparing read speeds for ratings (25m rows):\n")

t0 = time.time()
df_csv = pd.read_csv(DATA_DIR / "ratings.csv")
t_csv = time.time() - t0
print(f"csv read:     {t_csv:>5.2f}s")

t0 = time.time()
df_pq = pd.read_parquet(OUT_DIR / "ratings.parquet")
t_pq = time.time() - t0
print(f"parquet read: {t_pq:>5.2f}s")

print(f"\nspeedup: {t_csv / t_pq:.1f}x faster")

# bonus: read only specific columns from parquet (csv can't do this efficiently)
t0 = time.time()
df_subset = pd.read_parquet(OUT_DIR / "ratings.parquet", columns=["userId", "movieId"])
t_subset = time.time() - t0
print(f"\nparquet read (only userId, movieId): {t_subset:.2f}s")
print(f"shape: {df_subset.shape}, memory: {df_subset.memory_usage(deep=True).sum() / 1e6:.1f} mb")

comparing read speeds for ratings (25m rows):

csv read:      2.45s
parquet read:  3.13s

speedup: 0.8x faster

parquet read (only userId, movieId): 0.07s
shape: (25000095, 2), memory: 400.0 mb
